In [ ]:
#Fichier HELPER POUR DECOUPER LES FICHIERS ...ETC

In [1]:
import csv
import sys
import pandas as pd

In [2]:
csv.field_size_limit(sys.maxsize) 

131072

In [ ]:
#split data into 4 files for parallelization
filename = "./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata.csv"
with open(filename, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    rows = list(reader)

header, data = rows[0], rows[1:]
total = len(data)
chunk = total // 4 #split into 4 files
for i in range(4):
    
    start = i * chunk
    end = (i + 1) * chunk if i < 3 else total
    outname = f"tables_metadata_part{i+1}.csv"
    with open(outname, "w", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(data[start:end])

In [ ]:
#merge the files back into a single file
output_file = "./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_rep1.csv"
header_written = False

with open(output_file, 'w', newline='', encoding='utf-8') as out_csv:
    writer = None
    
    for file in range(1,5):
        with open(f"./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_part{file}.csv", 'r', newline='', encoding='utf-8') as in_csv:
            reader = csv.reader(in_csv)
            header = next(reader)
            if not header_written:
                writer = csv.writer(out_csv)
                writer.writerow(header)  # write header only once
                header_written = True
            
            # Write the rest of the rows
            for row in reader:
                writer.writerow(row)

with open(output_file, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    rows = list(reader)
header, data = rows[0], rows[1:]
total = len(data)    
print(f"Merged files into {output_file}, total nb lines= {total}")


In [12]:
#merged some files i had here , ignore

llm1=  pd.read_csv(f"./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_LLM_rep1_cleaned.csv")
llm2=  pd.read_csv(f"./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_LLM_rep2_cleaned.csv")

final= pd.merge(table_df, llm1[["context","llm_gen"]], on="context", how="inner") #merge both files to access predictions and data features
final= pd.merge(final, llm2[["context","llm_format"]], on="context", how="inner") #merge both files to access predictions and data features


final.to_csv(f"./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_train.csv", index=False)
print(len(final), len(llm1), len(llm2)) #should be equl to 1676 for final


1676 1679 1679


In [3]:
table_df= pd.read_csv(f"WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_test.csv")
llm1=  pd.read_csv(f"WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_test_part1.csv")
final= pd.merge(table_df, llm1[["context","llm_gen"]], on="context", how="inner") #merge both files to access predictions and data features
final.to_csv(f"./WikiTableQuestions-1.0.2-compact/WikiTableQuestions/tables_metadata_test.csv", index=False)

print(len(final), len(llm1), len(table_df)) #should be equl to 1676 for final

421 421 421
